In [ ]:
import json
import os
from io import StringIO
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from loguru import logger

from unga81.database import get_all_countries, get_db_connection
from unga81.config import EXTERNAL_DATA_DIR
from unga81.restcountries import download_country_data

pd.options.display.max_columns = None
pd.options.display.max_rows = None

load_dotenv()

API_KEY = os.getenv("API_KEY_RESTCOUNTRIES")

In [ ]:
conn = get_db_connection()
df = get_all_countries(conn=conn)
logger.info(f"{df.shape[0]} speeches in the database")

# RestcountriesAPI

In [ ]:
country_information_path = EXTERNAL_DATA_DIR / "restcountriesapi"
country_information_path.mkdir(parents=True, exist_ok=True)


for i, r in df.iterrows():
    if isinstance(r["iso_3"], str):
        download_country_data(alpha_3=r["iso_3"], output_path=country_information_path)

In [ ]:
# name_common = data_pais["data"]["objects"][0]["names"]["common"]
# name_official = data_pais["data"]["objects"][0]["names"]["official"]
# lat, lng = data_pais["data"]["objects"][0]["capitals"][0]["coordinates"]
# capital_name = data_pais["data"]["objects"][0]["capitals"][0]["name"]
alpha_3 = data_pais["data"]["objects"][0]["codes"]["alpha_3"]
# data_pais["data"]["objects"][0]["flag"]
alpha_3

In [ ]:
# TODO parse
# - flag
# - capital
# - area
# - timezones
# - currency
# - languages
# - borders
# - flag png
# - gini index
# - coat of arms

In [ ]:
pd.DataFrame(data_pais["data"]["objects"]).T

In [ ]:
def get_country_data(alpha_3: str):
    file_path = EXTERNAL_DATA_DIR / "restcountriesapi" / f"{alpha_3}.json"
    if not file_path.exists():
        return None

    with open(file_path, "r") as file:
        data = json.load(file)
    return data

In [ ]:
country_data = get_country_data("AGO")

In [ ]:
country_data["data"]["objects"][0]["flag"]["url_png"]
country_data["data"]["objects"][0]["area"]["kilometers"]
country_data["data"]["objects"][0]["borders"]
country_data["data"]["objects"][0]["economy"]["gini_coefficient"]
capitals = ", ".join(
    [x["name"] for x in country_data["data"]["objects"][0]["capitals"]]
)
country_data["data"]["objects"][0]["government_type"]
", ".join([x["name"] for x in country_data["data"]["objects"][0]["languages"]])
len(country_data["data"]["objects"][0]["languages"])

# Worldmeter

In [ ]:
def get_tables_from_website(url) -> list | None:
    response = requests.get(url)
    if response.ok:
        tables = pd.read_html(StringIO(response.text))
        logger.info(f"{len(tables)} tables found in {url}.")
        return tables
    return None


def parse_table(df: pd.DataFrame) -> pd.DataFrame: ...

In [ ]:
url_gdp_by_country = "https://www.worldometers.info/gdp/gdp-by-country/"
tables = get_tables_from_website(url_gdp_by_country)

In [ ]:
gdp_imf = tables[0]
gdp_wb = tables[1]

In [ ]:
gdp_imf = gdp_imf[["Country", "GDP (Full Value, US$)", "GDP per Capita (US$)"]]
gdp_imf.columns = ["Country", "GDP (US$) IMF", "GDP per Capita (US$) IMF"]
gdp_imf

In [ ]:
gdp_wb = gdp_wb[["Country", "GDP (Full Value, US$)", "GDP per Capita (US$)"]]
gdp_wb.columns = ["Country", "GDP (US$) WB", "GDP per Capita (US$) WB"]
gdp_wb

In [ ]:
gdp_imf.merge(gdp_wb, on="Country")

In [ ]:
for i, r in gdp_imf.head(10).iterrows():
    print(
        f'{r["Country"]} {r["GDP (Full Value, US$)"]} {int(r["GDP (Full Value, US$)"].replace(",", "").replace("$", "")):,}'
    )

In [ ]:
ulr_population = "https://www.worldometers.info/geography/countries-of-the-world/"
population_tables = get_tables_from_website(url=ulr_population)

In [ ]:
df_population = pd.concat([population_tables[0], population_tables[1]])
df_population[["Country", "Population 2026"]].reset_index(drop=True)

In [ ]:
url_energy = "https://www.worldometers.info/energy/energy-by-country/"
energy_tables = get_tables_from_website(url=url_energy)

In [ ]:
energy_tables[0]

In [ ]:
url_co2_emissions = (
    "https://www.worldometers.info/co2-emissions/co2-emissions-by-country/"
)
tables_co2 = get_tables_from_website(url=url_co2_emissions)

In [ ]:
tables_co2[0]

In [ ]:
url_ghg_emissions = "https://www.worldometers.info/greenhouse-gas-emissions/greenhouse-gas-emissions-by-country/"
tables_ghg = get_tables_from_website(url=url_ghg_emissions)

In [ ]:
tables_ghg[0]

# SDG Report

In [ ]:
url_sdg_ranking = "https://dashboards.sdgindex.org/rankings/"
tables_sdg = get_tables_from_website(url=url_sdg_ranking)

In [ ]:
for i, r in tables_sdg[0].iterrows():
    if "United" in r["Country"]:
        print(r["Country"])

In [ ]:
response = requests.get("https://dashboards.sdgindex.org/profiles/")
if response.ok:
    soup = BeautifulSoup(response.text, "html.parser")

In [ ]:
def get_sdg_index():
    response = requests.get("https://dashboards.sdgindex.org/profiles/")
    if response.ok:
        soup = BeautifulSoup(response.text, "html.parser")

        links = soup.find_all("a")
        data = []
        base_url = "https://dashboards.sdgindex.org/profiles/"
        for link in links:
            if (
                link.get("href").startswith("/profiles/")
                and len(
                    link.get("href").replace("/profiles/", "").replace("/", "").strip()
                )
                > 0
            ):
                country_name = link.text.strip()
                country_link = link.get("href").replace("/profiles/", "")
                data.append((country_name, base_url + country_link))
        df_sdg = pd.DataFrame(data, columns=["country", "sdg_profile"])

        # preprocessing data from the SDG report
        country_map = {
            "Bahamas, The": "Bahamas",
            "Cote d'Ivoire": "Côte d'Ivoire",
            "Egypt, Arab Rep.": "Egypt",
            "Kyrgyz Republic": "Kyrgyzstan",
            "Congo, Dem. Rep.": "Democratic Republic of the Congo",
            "Gambia, The": "Gambia",
            "Iran, Islamic Rep.": "Iran",
            "Korea, Dem. Rep.": "Republic of Korea",
            "Lao PDR": "Lao People's Democratic Republic",
            "Micronesia, Fed. Sts.": "Micronesia",
            "Slovak Republic": "Slovakia",
            "St. Kitts and Nevis": "Saint Kitts and Nevis",
            "St. Lucia": "Saint Lucia",
            "St. Vincent and the Grenadines": "Saint Vincent & Grenadines",
            "Syrian Arab Republic": "Syria",
            "Türkiye": "Turkey",
            "United States": "United States of America",
            "Venezuela, RB": "Venezuela",
            "Vietnam": "Viet Nam",
            "Yemen, Rep.": "Yemen",
        }

        for i, r in df_sdg.iterrows():
            if r["country"] in country_map:
                df_sdg.at[i, "country"] = country_map[r["country"]]

        return df_sdg

In [ ]:
df_sdg.head()

In [ ]:
# preprocessing data from the SDG report
country_map = {
    "Bahamas, The": "Bahamas",
    "Cote d'Ivoire": "Côte d'Ivoire",
    "Egypt, Arab Rep.": "Egypt",
    "Kyrgyz Republic": "Kyrgyzstan",
    "Congo, Dem. Rep.": "Democratic Republic of the Congo",
    "Gambia, The": "Gambia",
    "Iran, Islamic Rep.": "Iran",
    "Korea, Dem. Rep.": "Republic of Korea",
    "Lao PDR": "Lao People's Democratic Republic",
    "Micronesia, Fed. Sts.": "Micronesia",
    "Slovak Republic": "Slovakia",
    "St. Kitts and Nevis": "Saint Kitts and Nevis",
    "St. Lucia": "Saint Lucia",
    "St. Vincent and the Grenadines": "Saint Vincent & Grenadines",
    "Syrian Arab Republic": "Syria",
    "Türkiye": "Turkey",
    "United States": "United States of America",
    "Venezuela, RB": "Venezuela",
    "Vietnam": "Viet Nam",
    "Yemen, Rep.": "Yemen",
}

for i, r in df_sdg.iterrows():
    if r["country"] in country_map:
        df_sdg.at[i, "country"] = country_map[r["country"]]

In [ ]:
df.merge(df_sdg, how="outer", on="country")